# 🏥 Medical RAG System - Interactive Q&A
## Retrieval Augmented Generation for Medical Knowledge

This notebook implements a complete Medical RAG system with:
- **Semantic Search** using Sentence Transformers
- **BM25 Hybrid Retrieval** for better accuracy
- **LLM-based Generation** for comprehensive answers
- **Beautiful Gradio UI** for easy interaction

Perfect for Google Colab! ✨

## Cell 1: Install Dependencies

In [ ]:
# Install all required packages
import subprocess
import sys

packages = [
    'gradio',
    'sentence-transformers',
    'transformers',
    'torch',
    'scikit-learn',
    'nltk',
    'rank-bm25',
    'accelerate',
    'protobuf'
]

print('📦 Installing dependencies...')
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✅ All dependencies installed successfully!')

## Cell 2: Import Libraries

In [ ]:
import re
import json
import math
import time
import warnings
import numpy as np
from typing import List, Dict, Tuple
from collections import defaultdict

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

# Retrieval Libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# Generation & UI
from transformers import pipeline
import gradio as gr

warnings.filterwarnings('ignore')

# Download NLTK data
print('📥 Downloading NLTK data...')
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(resource, quiet=True)

print('✅ All imports completed!')

## Cell 3: Core RAG System Classes

In [ ]:
class MedicalDocumentProcessor:
    """Processes and cleans medical documents"""
    
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
    
    def clean_text(self, text: str) -> str:
        """Clean medical text"""
        text = re.sub(r'\s+', ' ', text.strip())
        text = re.sub(r'[^a-zA-Z0-9\s\-.,()]', '', text)
        return text
    
    def tokenize(self, text: str) -> List[str]:
        """Tokenize text into words"""
        tokens = word_tokenize(text.lower())
        return [t for t in tokens if t.isalnum() and t not in self.stop_words]
    
    def lemmatize_tokens(self, tokens: List[str]) -> List[str]:
        """Lemmatize tokens"""
        return [self.lemmatizer.lemmatize(token) for token in tokens]


class HybridRetriever:
    """Combines BM25 and semantic search for retrieval"""
    
    def __init__(self, use_semantic: bool = True):
        self.use_semantic = use_semantic
        self.bm25 = None
        self.documents = []
        self.semantic_model = None
        self.embeddings = None
        
        if use_semantic:
            print('⏳ Loading semantic search model...')
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print('✅ Semantic model loaded!')
    
    def add_documents(self, docs: List[str]):
        """Add documents to the retriever"""
        self.documents = docs
        
        tokenized_docs = [doc.lower().split() for doc in docs]
        self.bm25 = BM25Okapi(tokenized_docs)
        
        if self.use_semantic and self.semantic_model:
            print('⏳ Creating semantic embeddings...')
            self.embeddings = self.semantic_model.encode(docs, convert_to_tensor=False)
            print('✅ Embeddings created!')
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """Retrieve top-k documents using hybrid approach"""
        if not self.documents:
            return []
        
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_scores = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) - np.min(bm25_scores) + 1e-10)
        
        scores = bm25_scores.copy()
        
        if self.use_semantic and self.semantic_model:
            query_embedding = self.semantic_model.encode(query, convert_to_tensor=False)
            semantic_scores = cosine_similarity([query_embedding], self.embeddings)[0]
            scores = 0.4 * bm25_scores + 0.6 * semantic_scores
        
        top_indices = np.argsort(scores)[-top_k:][::-1]
        return [(self.documents[i], float(scores[i])) for i in top_indices]


class RAGAnswerer:
    """Generate answers using retrieved documents"""
    
    def __init__(self):
        print('⏳ Loading text generation model...')
        self.qa_pipeline = pipeline(
            'question-answering',
            model='deepset/roberta-base-squad2',
            device=0 if torch_available() else -1
        )
        print('✅ Generation model loaded!')
    
    def generate_answer(self, question: str, context: str) -> Dict:
        """Generate answer from question and context"""
        if not context.strip():
            return {'answer': '⚠️ No relevant documents found.', 'confidence': 0}
        
        try:
            result = self.qa_pipeline(question=question, context=context, max_answer_len=150)
            return result
        except Exception as e:
            return {'answer': f'Error: {str(e)}', 'confidence': 0}


def torch_available():
    """Check if CUDA is available"""
    try:
        import torch
        return torch.cuda.is_available()
    except:
        return False


print('✅ Core classes initialized!')

## Cell 4: Sample Medical Knowledge Base

In [ ]:
# Sample medical documents for demonstration
MEDICAL_DOCUMENTS = [
    "Diabetes mellitus is a metabolic disorder characterized by elevated blood glucose levels. Type 1 diabetes is an autoimmune condition where the pancreas fails to produce insulin. Type 2 diabetes occurs when the body becomes resistant to insulin. Management includes medication, diet control, and regular exercise. Complications include nephropathy, neuropathy, and retinopathy.",
    
    "Hypertension or high blood pressure is a chronic condition affecting millions. Normal blood pressure is less than 120/80 mmHg. Hypertension stages include elevated, Stage 1, and Stage 2. Risk factors include obesity, stress, alcohol, and lack of exercise. Treatment involves lifestyle changes and antihypertensive medications like ACE inhibitors and beta-blockers.",
    
    "Heart disease is the leading cause of death globally. Coronary artery disease occurs when plaque builds up in arteries. Symptoms include chest pain, shortness of breath, and fatigue. Risk factors include high cholesterol, hypertension, smoking, and diabetes. Prevention involves healthy lifestyle, medications like statins, and regular cardiac monitoring.",
    
    "Depression is a mental health disorder affecting mood, sleep, and appetite. Major depressive disorder involves persistent depressed mood for at least 2 weeks. Symptoms include anhedonia, fatigue, guilt, and concentration difficulties. Treatment options include SSRIs, cognitive behavioral therapy, and lifestyle modifications.",
    
    "Asthma is a chronic respiratory disease characterized by airway inflammation. Symptoms include wheezing, shortness of breath, chest tightness, and coughing. Triggers include allergens, exercise, cold air, and infections. Management uses inhaled corticosteroids and bronchodilators.",
    
    "Pneumonia is an infection causing lung inflammation and fluid in alveoli. Symptoms include cough, fever, dyspnea, and chest pain. Bacterial, viral, and fungal pneumonia require different treatments. Diagnosis involves chest X-ray. Treatment includes antibiotics and supportive care.",
    
    "Thyroid disorders affect metabolism and energy levels. Hypothyroidism results from insufficient thyroid hormone, causing fatigue and weight gain. Hyperthyroidism causes excessive hormone production, leading to anxiety and weight loss. TSH and free T4 levels guide diagnosis and treatment.",
    
    "Arthritis includes osteoarthritis and rheumatoid arthritis. Osteoarthritis involves cartilage degeneration from wear and tear. Rheumatoid arthritis is an autoimmune condition causing joint inflammation. Symptoms include pain, swelling, and stiffness. Treatment includes NSAIDs, physical therapy, and disease-modifying drugs.",
    
    "Fever is an elevated body temperature, usually above 37.5 degrees Celsius. Common causes include viral infections like flu and cold, bacterial infections like strep throat and urinary tract infections, and inflammatory conditions. Symptoms accompanying fever include chills, sweating, body aches, headache, and fatigue. Treatment includes rest, hydration, and antipyretics like paracetamol or ibuprofen. Most fevers resolve within a few days.",
    
    "Migraine is a neurological condition causing severe, throbbing headaches usually on one side of the head. Symptoms include light sensitivity, sound sensitivity, nausea, and vomiting. Triggers include stress, hormonal changes, certain foods, caffeine, and lack of sleep. Treatment includes pain relievers, triptans, and preventive medications.",
    
    "Influenza or flu is a contagious respiratory disease caused by influenza viruses. Symptoms include sudden onset fever, cough, body aches, fatigue, and sore throat. The flu is more severe than common cold. Complications include pneumonia. Vaccination is the primary prevention. Treatment includes antivirals and supportive care.",
    
    "Gastroenteritis or stomach flu is inflammation of the stomach and intestines from viral or bacterial infection. Symptoms include vomiting, diarrhea, abdominal cramps, and loss of appetite. Rotavirus and norovirus are common viral causes. Management focuses on hydration with oral rehydration solutions. Most cases resolve within days.",
    
    "Urinary tract infections occur when bacteria infect the bladder or urethra. Symptoms include burning during urination, frequent urination, cloudy urine, and lower abdominal pain. Women are more susceptible. Diagnosis involves urinalysis and urine culture. Treatment includes antibiotics. Prevention involves proper hygiene and hydration."
]

## Cell 5: Initialize RAG System

In [ ]:
print('🚀 Initializing Medical RAG System...')
print('=' * 50)

# Initialize components
processor = MedicalDocumentProcessor()
retriever = HybridRetriever(use_semantic=True)
answerer = RAGAnswerer()

# Add documents to retriever
print('\n📖 Loading medical documents into knowledge base...')
retriever.add_documents(MEDICAL_DOCUMENTS)

print('\n✅ RAG System ready for queries!')
print('=' * 50)

## Cell 6: Query Function

In [ ]:
def rag_query(question: str, top_k: int = 3) -> Dict:
    """Execute RAG pipeline"""
    
    if not question.strip():
        return {
            'answer': '❌ Please enter a question',
            'confidence': 0,
            'sources': []
        }
    
    # Step 1: Retrieve relevant documents
    retrieved_docs = retriever.retrieve(question, top_k=top_k)
    
    if not retrieved_docs:
        return {
            'answer': '⚠️ No relevant documents found in knowledge base',
            'confidence': 0,
            'sources': []
        }
    
    # Step 2: Combine context
    context = '\n'.join([doc for doc, score in retrieved_docs])
    
    # Step 3: Generate answer
    result = answerer.generate_answer(question, context)
    
    # Format sources
    sources = [f"📄 [{score:.2%}] {doc[:80]}..." for doc, score in retrieved_docs]
    
    return {
        'answer': result.get('answer', 'Unable to generate answer'),
        'confidence': round(result.get('score', 0) * 100, 2),
        'sources': sources
    }

print('✅ Query function ready!')

## Cell 7: Beautiful Gradio Interface

In [ ]:
def create_interface():
    """Create attractive Gradio interface"""
    
    with gr.Blocks(
        title='🏥 Medical RAG System',
        theme=gr.themes.Soft(primary_hue='blue')
    ) as interface:
        
        # Header
        gr.Markdown(
            """
            # 🏥 Medical Q&A RAG System
            
            **Intelligent Question Answering using Retrieval Augmented Generation**
            
            Ask any medical question and get evidence-based answers from our knowledge base!
            """
        )
        
        with gr.Row():
            with gr.Column(scale=2):
                question = gr.Textbox(
                    label='❓ Medical Question',
                    placeholder='E.g., What are the symptoms of diabetes?',
                    lines=2
                )
                
                with gr.Row():
                    top_k = gr.Slider(
                        label='📚 Documents to Retrieve',
                        minimum=1,
                        maximum=5,
                        value=3,
                        step=1
                    )
                    submit_btn = gr.Button(
                        '🔍 Search',
                        variant='primary',
                        size='lg'
                    )
            
            with gr.Column(scale=1):
                gr.Markdown(
                    """
                    ### 💡 Example Questions
                    
                    - What is diabetes?
                    - How is hypertension treated?
                    - What causes asthma?
                    - Symptoms of depression?
                    - Heart disease prevention
                    """
                )
        
        # Results section
        with gr.Group():
            gr.Markdown('## 📋 Results')
            
            answer_output = gr.Markdown(
                label='Answer',
                value='💬 Answers will appear here...'
            )
            
            with gr.Row():
                confidence = gr.Number(
                    label='📊 Confidence Score',
                    interactive=False
                )
            
            sources = gr.Markdown(
                label='Sources',
                value='📚 Source documents will appear here...'
            )
        
        # Info section
        gr.Markdown(
            """
            ---
            ### ℹ️ How it works
            1. **Retrieval**: Searches knowledge base using BM25 + Semantic Search
            2. **Ranking**: Combines BM25 (40%) + Semantic similarity (60%)
            3. **Generation**: Uses RoBERTa-based QA model to extract answers
            4. **Display**: Shows answer with confidence and source documents
            """
        )
        
        # Event handler
        def process_query(q, k):
            result = rag_query(q, top_k=int(k))
            
            answer_md = f"### 💬 Answer\n\n{result['answer']}"
            sources_md = f"### 📚 Source Documents\n\n" + '\n\n'.join(result['sources']) if result['sources'] else "No sources found"
            
            return answer_md, result['confidence'], sources_md
        
        submit_btn.click(
            process_query,
            inputs=[question, top_k],
            outputs=[answer_output, confidence, sources]
        )
        
        # Example questions
        example_questions = [
            ['What are the symptoms of diabetes?', 3],
            ['How is hypertension treated?', 3],
            ['What causes heart disease?', 3]
        ]
        
        gr.Examples(
            examples=example_questions,
            inputs=[question, top_k],
            outputs=[answer_output, confidence, sources],
            fn=process_query,
            cache_examples=False
        )
    
    return interface


print('✅ Interface created!')

## Cell 8: Launch Interface

In [ ]:
# Create and launch the interface
interface = create_interface()

# For Google Colab
interface.launch(
    share=True,  # Creates public link
    show_error=True,
    show_api=False,
    quiet=False
)

## Cell 9: Optional - Test Queries (Without UI)

In [ ]:
# Run some test queries
test_questions = [
    'What are the symptoms of diabetes?',
    'How do you treat hypertension?',
    'What causes asthma?'
]

print('🧪 Running test queries...\n')
print('=' * 70)

for q in test_questions:
    print(f'\n❓ Question: {q}')
    result = rag_query(q, top_k=2)
    print(f'\n💬 Answer: {result["answer"]}')
    print(f'📊 Confidence: {result["confidence"]}%')
    print('\n📚 Sources:')
    for source in result['sources']:
        print(f'  {source}')
    print('\n' + '-' * 70)

print('\n✅ Test queries completed!')